In [1]:
import sys

!{sys.executable} -m pip install sentence-transformers

In [1]:
from sentence_transformers import SentenceTransformer

MODEL_NAME = "BAAI/bge-m3"
model = SentenceTransformer(MODEL_NAME)

print("BGE-M3 다운로드 및 로드 완료")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BGE-M3 다운로드 및 로드 완료


In [2]:
import sys

!{sys.executable} -m pip install chonkie

  Using cached chonkie-1.6.8-py3-none-any.whl.metadata (29 kB)
   ---------------------------------------- 0.0/2.3 MB ? eta -:--:--
   --------------------------- ------------ 1.6/2.3 MB 9.1 MB/s eta 0:00:01
   ---------------------------------------- 2.3/2.3 MB 10.5 MB/s eta 0:00:00

   ------------- -------------------------- 1/3 [chonkie-core]
   -------------------------- ------------- 2/3 [chonkie]
   -------------------------- ------------- 2/3 [chonkie]
   -------------------------- ------------- 2/3 [chonkie]
   -------------------------- ------------- 2/3 [chonkie]
   -------------------------- ------------- 2/3 [chonkie]
   -------------------------- ------------- 2/3 [chonkie]
   -------------------------- ------------- 2/3 [chonkie]
   -------------------------- ------------- 2/3 [chonkie]
   -------------------------- ------------- 2/3 [chonkie]
   -------------------------- ------------- 2/3 [chonkie]
   -------------------------- ------------- 2/3 [chonkie]
   ----------

In [1]:
import chonkie
print(chonkie.__version__)

1.6.8


In [2]:
# 버전 확인 및 임포트
import chonkie, transformers, tokenizers, sentence_transformers
print("chonkie:", chonkie.__version__)
print("transformers:", transformers.__version__)
print("tokenizers:", tokenizers.__version__)
print("sentence_transformers:", sentence_transformers.__version__)

chonkie: 1.6.8
transformers: 5.12.1
tokenizers: 0.22.2
sentence_transformers: 5.6.0


In [4]:
import json
from chonkie import SemanticChunker, SentenceTransformerEmbeddings

MODEL = "BAAI/bge-m3"        # huggingface에서 모델+토크나이저 전체 다운로드 (약 2GB)
INPUT_PATH = "C:\\Users\\PC\\Documents\\카카오톡 받은 파일\\생성형 AI 활용 시책제안 및 우수사례 공모전 추진계획(통보용).md"         # 처리할 실제 md 파일 경로
OUTPUT_PATH = "C:\\Users\\PC\\Documents\\카카오톡 받은 파일\\chunks.json"    

THRESHOLD = 0.3    # 의미 경계 민감도(0~1). 높을수록 덜 자름(큰 청크), 낮을수록 잘게.
CHUNK_SIZE = 1024  # 청크 최대 토큰 수 (이 한도 넘으면 강제로 끊음)
MIN_SENTENCES = 2  # 한 청크가 최소한 가져야 하는 문장 개수

## 임베딩 모델 로드 (모델+토크나이저 전체 다운로드, 약 2GB, huggingface 계정 불필요)

In [5]:
embeddings = SentenceTransformerEmbeddings(MODEL)
print("BGE-M3 로드 완료, 임베딩 차원:", embeddings.dimension)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BGE-M3 로드 완료, 임베딩 차원: 1024


C:\Anaconda\Lib\site-packages\chonkie\embeddings\sentence_transformer.py:62: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self._dimension = self.model.get_sentence_embedding_dimension()


## 청커 생성 + 청킹

In [6]:
chunker = SemanticChunker(
    embedding_model=embeddings,
    threshold=THRESHOLD,
    chunk_size=CHUNK_SIZE,
    min_sentences=MIN_SENTENCES,
)

with open(INPUT_PATH, encoding="utf-8") as f:
    text = f.read()

chunks = chunker(text)
print(f"→ {len(chunks)}개 청크 생성")

→ 11개 청크 생성


## 미리보기

In [7]:
for i, c in enumerate(chunks[:5]):
    preview = c.text.strip().replace("\n", " ")[:60]
    print(f"[{i}] (토큰≈{c.token_count}) {preview}")

[0] (토큰≈101) # 생성형 AI활용 시책 제안 및  # 업무활용 우수사례 공모전 추진계획  (기획예산과)  # 생성형 AI활
[1] (토큰≈48) 급격한 발전에 따라 행정 업무의 패러다임 전환 필요  ☐ (AI 역량강화) AI활용 사례를 적극 발굴하여 직
[2] (토큰≈27) ☐ 공모기간 : 2025. 8. 18.(월) \~ 8. 29.(금) / 2주간  ☐ 제
[3] (토큰≈39) 출 : 담당자 온-메일 제출  ☐ 참가대상 : 고령군 소속 직원  ☐ 공모분야 : ①시책 제안(군정 전 분야
[4] (토큰≈71) - ○ (시책제안) 생성형 AI를 활용하여 군정 발전을 이끌 수 있는 정책 아이디어 제시(군정 전 분야)  


In [8]:
## JSONL 저장

In [9]:
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    for i, c in enumerate(chunks):
        record = {"chunk_id": i, "text": c.text.strip(), "token_count": c.token_count}
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print(f"저장 완료: {OUTPUT_PATH}")

저장 완료: C:\Users\PC\Documents\카카오톡 받은 파일\chunks.json
